# OpenFF Workshop: Simulating Post-Translationally Modified Proteins with the OpenFF Rosemary Alpha

**Run this notebook yourself!** https://github.com/openforcefield/2026-virtual-workshops/

<img src="qr-code.png" width=250 style="margin: auto" />

**Open Force Field (OpenFF)** is an industry-funded, open-source consortium that builds classical force fields and the software to use them. This workshop was developed by Ashley Mitchell from the OpenFF infrastructure team.

**Today's goal:** show how to apply the release candidate of our protein + small molecule force field (**Rosemary**) to post-translationally modified proteins.

### Roadmap

| Part | What we'll do | The new idea |
|---|---|---|
| 1 | Simulate a small molecule | The basic workflow: from a SMILES string to a simulation in about 10 lines |
| 2 | Watch lipids self-assemble into a micelle | Building a box containing many molecules |
| 3 | Load a "stapled" peptide drug from a PDB file | Handling chemistry that standard PDB tools can't |
| 4 | Redo the micelle with Amber lipid parameters | Mixing OpenFF with other force fields |
| 5 | Semaglutide (Ozempic) bound to its receptor in a membrane | All of the above at once |

> The presenter's copy has already been run, because some simulations take several minutes. You can just watch; you don't need to run anything live.

### Molecular dynamics in one minute

- **Molecular dynamics (MD)** approximates the physics of a system using classical, Newtonian mechanics. It calculates the force on every atom, moves every atom forward by a tiny time step (a few femtoseconds), and repeats this millions of times. The resulting **trajectory** shows how the system behaves.
- A **force field** is the mathematical description of those forces. **This is what OpenFF makes:** the parameter files and the software that applies them.

**Every example today follows the same three steps:**

```
  1. TOPOLOGY            2. PARAMETRIZE              3. SIMULATE
  "what's in the box"  → apply a force field      →  run MD in an engine
  (molecules, bonds)     (creates an Interchange)    (OpenMM here)
```

### Cheat sheet

| Term | Meaning |
|---|---|
| **Topology** | The chemical description of what we simulate: molecules, atoms, bonds, and optionally positions and a box |
| **Force field** | Parameters and equations that give the energy of, and forces on, every atom |
| **Parametrize** | Assign force field parameters to every atom, bond, angle, and so on |
| **Interchange** | OpenFF's parametrized system. It can import from and export to many simulation engines |
| **SMILES / SMIRKS** | SMILES is a text string that describes a molecule. SMIRKS is a text string that describes a chemical *pattern* |
| **`.offxml`** | OpenFF's force field file format, which assigns parameters by matching SMIRKS patterns |
| **Sage (2.x)** | OpenFF's small molecule force field |
| **Rosemary (3.x)** | OpenFF's upcoming force field for proteins and small molecules (in alpha) |
| **PDB** | The most common file format for protein structures. It has coordinates but incomplete chemistry |
| **Non-canonical amino acid / PTM** | An amino acid outside the standard 20, or a chemical change to a protein after it is made (post-translational modification) |
| **HMR** | Hydrogen mass repartitioning: making hydrogens heavier so we can take larger time steps |
| **OpenMM** | The open-source simulation engine we run today |
| **nglview** | The 3D molecule viewer used in this notebook |

## Part 1: The basic workflow

We'll simulate **acetaminophen** (paracetamol), starting from nothing but its SMILES string.

> **About `.offxml`:** most force fields assign parameters through *atom types*, which are labels a human gives to each atom. OpenFF force fields skip atom types and assign parameters directly from the chemistry by matching SMIRKS patterns.

In [ ]:
from openff.toolkit import ForceField, Molecule, Topology

# Step 1. Topology
molecule = Molecule.from_smiles("CC(=O)Nc1ccc(cc1)O")
molecule.generate_conformers(n_conformers=1)
topology = Topology.from_molecules([molecule])

# Step 2. Parametrize
force_field = ForceField("openff-2.3.0.offxml")  # NB: This is not the Rosemary alpha
interchange = force_field.create_interchange(topology)
interchange.minimize()

# 3. Hand off to simulation engine
openmm_system = interchange.to_openmm_system()
openmm_topology = interchange.to_openmm_topology()
openmm_positions = interchange.positions.to_openmm()

**Step 3: Simulate.** This is plain OpenMM code with nothing OpenFF-specific in it. It runs 10,000 steps × 2 fs = **20 picoseconds** of one molecule in vacuum.

In [ ]:
# This cell is all OpenMM code
import openmm

temperature = 298.15 * openmm.unit.kelvin
friction_coefficient = 1.0 / openmm.unit.picosecond
step_size = 2.0 * openmm.unit.femtosecond

simulation = openmm.app.Simulation(
    openmm_topology,
    openmm_system,
    openmm.LangevinIntegrator(temperature, friction_coefficient, step_size),
)
simulation.context.setPositions(openmm_positions)
simulation.context.setVelocitiesToTemperature(simulation.integrator.getTemperature())

simulation.reporters.append(
    openmm.app.DCDReporter(file="trajectory.dcd", reportInterval=100)
)
simulation.step(10000)

Load the trajectory and play it.

> **Watch for:** the molecule vibrating and rotating around its bonds. It's one molecule in a vacuum, so expect nothing dramatic.

In [ ]:
import mdtraj
import nglview

trajectory: mdtraj.Trajectory = mdtraj.load(
    "trajectory.dcd", top=mdtraj.Topology.from_openmm(interchange.to_openmm_topology())
)

view = nglview.show_mdtraj(trajectory)
view

## What just happened?

We went from a SMILES string to a running simulation in about 10 lines of code:

```python
from openff.toolkit import ForceField, Molecule, Topology

molecule = Molecule.from_smiles("CC(=O)Nc1ccc(cc1)O")
molecule.generate_conformers(n_conformers=1)

topology = Topology.from_molecules([molecule])

force_field = ForceField("openff-2.3.0.offxml")
interchange = force_field.create_interchange(topology)
interchange.minimize()

openmm_system = interchange.to_openmm_system()
openmm_topology = interchange.to_openmm_topology()
openmm_positions = interchange.positions.to_openmm()
```

1. **Easy**: Minimized OpenMM system ready in 10 lines and 5 seconds starting from nothing but a SMILES string and some FOSS
2. **Explicit**: All force field parameters are explicit in the force field file, including PME, cutoffs, functional form, constraints...
3. **Chemical**: System construction and parametrization is based on chemistry, not particles and atom types, and chemical identities survive parametrization
4. **Universal**: System construction and parametrization is the same across all supported simulation engines - OpenMM, Amber, GROMACS, and LAMMPS
5. **Extensible**: All steps are modular and can be augmented with arbitrary Python code

> **Pause for questions.** Next come more realistic systems, but they all use these same three steps.

## A helper for the rest of the workshop

`simulate_and_visualize()` packages the OpenMM code from Part 1 so we don't retype it for every example. It also adds energy minimization, optional pressure control, and hydrogen mass repartitioning.

**You don't need to read this cell.** All it does is:

1. Export the Interchange to OpenMM
2. Minimize the energy
3. Run MD
4. Return a 3D viewer of the trajectory

In [ ]:
from datetime import datetime
from tempfile import NamedTemporaryFile

import mdtraj
import nglview
import openmm
from openff.interchange import Interchange
from openff.toolkit import ForceField, Molecule, Quantity, Topology

HYDROGEN_MASS = Quantity(1.007947, "amu")


def simulate_and_visualize(
    interchange: Interchange,
    steps: int = 10_000,
    temperature: Quantity = Quantity(298.15, "K"),
    friction_coefficient: Quantity = Quantity(1.0, "1/ps"),
    step_size: Quantity = Quantity(2.0, "fs"),
    hydrogen_mass: Quantity = HYDROGEN_MASS,
    pressure: None | Quantity = None,
    membrane_surface_tension: None | Quantity = None,
    report_interval: int = 100,
) -> nglview.NGLWidget:
    """
    Convenience function for running and visualizing a brief simulation.

    A Langevin integrator is used, and so temperature coupling is required.
    Parameters for configuring Hydrogen mass repartitioning and pressure
    coupling are available. Pressure coupling may be performed isotropically
    or in a scheme appropriate for a membrane spanning the X-Y plane of the
    box. By default, performs a 20 picosecond constant NVT simulation at
    25° C.

    The trajectory is stored in a temporary file which is cleaned up after
    the visualization is prepared.

    Parameters
    ==========
    interchange
        The Interchange object to simulate.
    steps
        The number of MD steps to take.
    temperature
        The temperature for the MD simulation. Velocities are generated at
        this temperature and a Langevin integrator is used to maintain it.
    friction_coefficient
        The friction coefficient for the Langevin integrator.
    step_size
        The duration of an MD time step.
    hydrogen_mass
        The mass of hydrogen atoms in the simulation. This may be used to
        implement a hydrogen mass repartitioning (HMR) scheme. Mass added
        to hydrogen is removed from the neighbouring atoms so that the
        net mass of the system is constant. The default is no mass
        repartitioning.
    pressure
        The pressure for a Monte Carlo barostat. The default is no
        barostat (constant volume simulation). If a pressure is provided,
        the ``membrane_surface_tension`` parameter determines whether
        isotropic pressure coupling is used, or if the Z dimension is
        treated separately to the X and Y dimensions.
    membrane_surface_tension
        Surface tension for a Monte Carlo barostat appropriate for membrane
        simulations. If no surface tension is provided (the default), the
        barostat is isotropic. If a surface tension is provided, then the
        Z dimension is treated separately from the X and Y dimensions, which
        are treated isotropically, and the surface tension applies to the
        X and Y dimensions. This is appropriate for a membrane simulation
        where the membrane spans the X-Y plane of the box.
    report_interval
        The number of steps between visualized MD frames.
    """
    print(f"{datetime.now()} Exporting...")
    # OpenMM implements its barostats as forces
    additional_forces = []
    if pressure is not None:
        if membrane_surface_tension is None:
            barostat = openmm.MonteCarloBarostat(
                pressure.to_openmm(),
                temperature.to_openmm(),
            )
        else:
            barostat = openmm.MonteCarloMembraneBarostat(
                pressure.to_openmm(),
                membrane_surface_tension.to_openmm(),
                temperature.to_openmm(),
                openmm.MonteCarloMembraneBarostat.XYIsotropic,
                openmm.MonteCarloMembraneBarostat.ZFree,
            )
        additional_forces.append(barostat)

    simulation = interchange.to_openmm_simulation(
        integrator=openmm.LangevinIntegrator(
            temperature.to_openmm(),
            friction_coefficient.to_openmm(),
            step_size.to_openmm(),
        ),
        hydrogen_mass=hydrogen_mass.m_as("amu"),
        additional_forces=additional_forces,
    )
    print(f"{datetime.now()} Minimizing...")
    simulation.minimizeEnergy()

    simulation.context.setVelocitiesToTemperature(
        simulation.integrator.getTemperature()
    )

    with NamedTemporaryFile(suffix=".dcd") as f:
        simulation.reporters.append(
            openmm.app.DCDReporter(file=f.name, reportInterval=report_interval)
        )
        print(f"{datetime.now()} Simulating...")
        simulation.step(steps)
        print(f"{datetime.now()} Visualizing.")
        trajectory: mdtraj.Trajectory = mdtraj.load(
            f.name, top=mdtraj.Topology.from_openmm(interchange.to_openmm_topology())
        )
    return nglview.show_mdtraj(trajectory)

## Part 2: A self-assembling micelle

**New idea: building a box with many molecules.**

Real systems contain thousands of molecules, and something has to place them all in a box. **Packmol** packs molecules randomly into a box to make a starting structure. Packmol isn't a Python package, but Interchange provides a convenient Python wrapper, `pack_box`.

We'll pack **25 DLPC lipids** and **6,000 waters** at 1 g/mL. The whole box is a single Topology.

> **Watch for:** a random jumble of lipids and water. That's the starting point.

In [ ]:
from openff.interchange.components._packmol import UNIT_CUBE, pack_box
from openff.toolkit import ForceField, Molecule, Quantity

dlpc = Molecule.from_smiles(
    "CCCCCCCCCCCC(=O)OC[C@H](CO[P@](=O)([O-])OCC[N+](C)(C)C)OC(=O)CCCCCCCCCCC"
)
water = Molecule.from_smiles("O")
for atom in water.atoms:
    atom.metadata["residue_name"] = "HOH"

micelle_top = pack_box(
    [dlpc, water],
    [25, 6000],
    target_density=Quantity(1.0, "g/mL"),
    box_shape=UNIT_CUBE,
    tolerance=Quantity(0.05, "nm"),
)
micelle_top.visualize()

**Step 2: Parametrize** everything with Sage 2.3. Sage includes TIP3P water parameters, so one file covers both the lipids and the water.

In [ ]:
sage_ff = ForceField("openff-2.3.0.offxml")
micelle_interchange = sage_ff.create_interchange(micelle_top)

**Step 3: Simulate** 50,000 steps × 4 fs = **0.2 nanoseconds**. HMR (heavier hydrogens) lets us use a 4 fs time step instead of 2 fs, which halves the cost. Water is hidden so you can see the lipids.

> **Watch for:** the greasy lipid tails clustering together in the middle while the water-loving heads face outward. That's a micelle forming on a laptop in a few minutes.

In [ ]:
w = simulate_and_visualize(
    micelle_interchange,
    steps=50_000,
    # steps=500_000, # For a really clear self assembly
    report_interval=10_000,
    hydrogen_mass=3 * HYDROGEN_MASS,  # HMR for larger step size
    step_size=Quantity(4.0, "fs"),
)
w.clear_representations()
w.add_representation("licorice", selection="not water")
w.add_representation("unitcell")
w

## Part 3: Loading a non-canonical peptide from a PDB file

**New idea: getting complete chemistry out of a PDB file** with OpenFF's **next-generation PDB loader**.

**Upcoming**

OpenFF 3 Rosemary

- Self consistent protein and drug-like small molecule force field
- Fast, scalable, conformation independent, Amber-compatible graph neural network charge model
- Same model treats proteins, peptides, small molecules, and post-translational modifications
- Now in early Alpha!

Next-Gen PDB loader

- Explicit and extensible residue library
- Supports fast name-based matching and slower graph-based matching
- Errors rather than guesses

### Why is this hard?

- To apply our force field we need **complete chemistry**: every atom including hydrogens, every bond, bond orders, and formal charges.
- PDB files were designed for **coordinates**. They mostly imply the chemistry through residue names, and they often lack hydrogens.
- So the loader matches each residue against an **explicit residue library**. Matching by name is fast. Matching by bonds (CONECT records) is slower but works for unusual residues.
- If it can't be sure, it **raises an error** instead of guessing.

> **About the name:** while it's experimental, the loader lives in a package called `openff.pablo`, which you'll see in the imports. It is planned to move into the main OpenFF Toolkit in early 2027.

### Our test case: sulanemadlin

- Experimental chemotherapy that mimics part of the p53 protein
- A hydrocarbon **staple** joins two side chains and locks the peptide into a helix
- That crosslink between two residues is exactly what standard PDB tools struggle with

In [ ]:
Molecule.from_smiles(
    "CC(=O)N[C@@H](CC(C)C)C(=O)N[C@H](C(=O)N[C@@H](Cc1ccccc1)C(=O)N[C@]1(C)CCCCCCCCCCC[C@@](C)(C(=O)N[C@@H](C)C(=O)N[C@@H](C)C(=O)N[C@@H](C)C(=O)N[C@@H](C)C(=O)N[C@@H](C)C(=O)N[C@H](C)C(=O)O)NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CCC(N)=O)NC(=O)[C@H](C)NC(=O)[C@H](Cc2c[nH]c3ccccc23)NC(=O)[C@H](Cc2ccc(O)cc2)NC(=O)[C@H](CCC(=O)[O-])NC1=O)[C@@H](C)O"
).visualize("rdkit", show_all_hydrogens=False)

The two stapled residues are defined in the **Chemical Component Dictionary (CCD)**, the PDB's official library of residue definitions. The loader keeps a cache on disk and downloads any residue it doesn't have yet (`auto_download = True`). Here are the two residues:

In [ ]:
# Currently in the experimental openff.pablo package; coming to the Toolkit soon
from openff.pablo import STD_CCD_CACHE
STD_CCD_CACHE.auto_download = True

display(STD_CCD_CACHE["0EH"][0].visualize(), STD_CCD_CACHE["MK8"][0].visualize())

**First attempt: load the PDB as-is.**

> ⚠️ **This cell is expected to raise an error.** It's a long error message, but the cause is simple, as explained below.

In [ ]:
from openff.pablo import topology_from_pdb

ptm_top = topology_from_pdb("8gjs_prepared.pdb")

**What went wrong?** The loader knows both staple residues, but it doesn't know they're **bonded to each other**.

- In the library, each residue is defined as a standalone molecule, with hydrogens where the crosslink would go.
- In the protein, those hydrogens are gone (they are **leaving atoms**) and a bond joins the two residues.
- So the atoms in the file don't match the library definition, and the loader refuses to guess.

A residue definition includes its atoms, bonds, and which atoms leave when it links to a neighbor:

<img src="./residue_definition.png" alt="Diagram of a residue definition" width="1200">

**The fix: tell the library about the crosslink.**

`with_crosslink` returns a *copy* of the residue library with one extra rule: bond atom `CAT` of residue `0EH` to atom `CE` of residue `MK8` with a single bond, and remove hydrogens `H22` and `HEB`. Now the file loads cleanly.

> You can keep growing a library like this. If you reuse one often, put it in its own Python module.

In [ ]:
from openff.pablo.residue import BondDefinition

ptm_top = topology_from_pdb(
    "8gjs_prepared.pdb",
    residue_library=STD_CCD_CACHE.with_crosslink(
        residues=["0EH", "MK8"],
        linking_atoms=["CAT", "CE"],
        leaving_atoms=[["H22"], ["HEB"]],
        bond_order=1,
    ),
)

The peptide (chain B, orange, atoms shown) sitting in its binding site on the target protein (chain A, dark red):

In [ ]:
w = ptm_top.visualize()
w.clear_representations()
w.add_representation("cartoon", selection=":A", color="#990000")
w.add_representation("cartoon", selection=":B", color="#FF7733")
w.add_representation("licorice", selection="not water and :B")
w.add_representation("unitcell")
w.add_spacefill(selection="water or ion", opacity=0.05)
w

Check that the chemistry survived by drawing the peptide in 2D. The staple is there:

In [ ]:
ptm_top.molecule(1).visualize("rdkit", show_all_hydrogens=False)

**Parametrize with the Rosemary alpha**, plus OPC3 water.

- `openff_no_water-3.0.0-alpha0.offxml` contains no water parameters, so we load OPC3 from a second file. When you load several files, parameters from later files take precedence.
- Rosemary's graph neural network charge model makes charge assignment fast even for big molecules.

In [ ]:
force_field = ForceField("openff_no_water-3.0.0-alpha0.offxml", "opc3.offxml")
ptm_interchange = force_field.create_interchange(ptm_top)

Simulate for 20 ps (the default settings) and watch the peptide move in its binding site.

In [ ]:
w = simulate_and_visualize(ptm_interchange)
w.clear_representations()
w.add_representation("cartoon", selection=":A", color="#990000")
w.add_representation("cartoon", selection=":B", color="#FF7733")
w.add_representation("licorice", selection="not water and :B")
w.add_representation("unitcell")
w.add_spacefill(selection="water or ion", opacity=0.05)
w

## Part 4: Mixing force fields with Amber lipids in the micelle

**New idea: combining parameters from different force fields** using `Interchange.from_openmm()` and `Interchange.combine()`.

Sometimes you want an established force field for part of your system. Interchange can *import* systems as well as export them, so we can:

1. Split the Part 2 micelle into lipids and everything else
2. Parametrize the water with **Sage** through OpenFF
3. Parametrize the lipids with **Amber Lipid17** through OpenMM, using the PME and cutoff settings that Lipid17 expects
4. Import the lipids back into an Interchange with the OpenFF topology, which keeps the full chemical information
5. **Combine** the two Interchanges into one system

> The picture won't change: the molecules are the same, only the parameters are different.

In [ ]:
import openmm
from openff.interchange import Interchange
from openff.toolkit import Topology

# 1. Split the topologies based on which force field we want for each molecule
lipid_molecules, solvent_molecules = [], []
for mol in micelle_top.molecules:
    if mol == dlpc:
        lipid_molecules.append(mol)
    else:
        solvent_molecules.append(mol)
lipid_top = Topology.from_molecules(lipid_molecules)
solvent_top = Topology.from_molecules(solvent_molecules)

lipid_top.box_vectors = micelle_top.box_vectors
solvent_top.box_vectors = micelle_top.box_vectors

# 2. Parametrize the non-lipid molecules into an Interchange
solvent_interchange = sage_ff.create_interchange(solvent_top)

# 3. Parametrize the lipids into an OpenMM System
lipid_system = openmm.app.ForceField("amber14/lipid17.xml").createSystem(
    lipid_top.to_openmm(),
    nonbondedMethod=openmm.app.PME,
    nonbondedCutoff=9 * openmm.unit.angstrom,
    switchDistance=8 * openmm.unit.angstrom,
    constraints=openmm.app.HBonds,
)

# 4. Import the parameters from the lipid OpenMM system, using the OpenFF topology
lipid_interchange = Interchange.from_openmm(lipid_system, lipid_top)

# 5. Combine the separately parametrized interchanges
combined_interchange = solvent_interchange.combine(lipid_interchange)

# Inspect visually
combined_interchange.visualize()

Simulate 250,000 steps × 4 fs = **1 nanosecond** (about 12 minutes on a MacBook Air).

> **Watch for:** with this longer run the micelle forms more quickly and more convincingly.

In [ ]:
w = simulate_and_visualize(
    combined_interchange,
    steps=250_000,
    # steps=500_000, # For a really clear self assembly
    report_interval=10_000,
    hydrogen_mass=3 * HYDROGEN_MASS,  # HMR for larger step size
    step_size=Quantity(4.0, "fs"),
)
w.clear_representations()
w.add_representation("licorice", selection="not water")
w.add_representation("unitcell")
w

## Part 5: Putting it all together with semaglutide bound to its receptor in a membrane

This part uses everything so far: PDB loading with custom chemistry (Part 3), Rosemary (Part 3), and Amber lipids combined with OpenFF (Part 4).

**Semaglutide**

![](semaglutide.svg)

- Brand names: Ozempic, Wegovy
- Activates the GLP-1 receptor
- Blockbuster weight loss and diabetes drug
- Alpha-helical peptide with **two non-canonical residues**:
    1. **Aib** (2-aminoisobutyric acid): alanine with an extra methyl group in place of its alpha hydrogen. It makes the peptide more helical.
    2. **Lysine with a long fatty chain attached.** The chain lets the drug bind to proteins in the blood, which makes it last much longer.
- The GLP-1 receptor sits in the cell membrane, so we need a membrane too.

### What the loading cell has to handle

| Challenge | Solution |
|---|---|
| Aib residue | Already in the standard library, so nothing extra is needed |
| Lysine bonded to the fatty chain (residue `KUT` in the CCD) | A crosslink rule, exactly as in Part 3 |
| POPC membrane lipids with non-standard residue names (the membrane was built with OpenMM) | `additional_definitions`: give the loader the lipid's SMILES and let it work out the match from the bonds. This is slower and experimental, but much less work than writing a full mapping of atom names |

> **How was the PDB prepared?** The original structure was missing half of the fatty chain. It was fixed with OpenMM Modeller and PDBFixer (for the membrane), plus hand editing. Structure preparation is still a weak spot across the field.

In [ ]:
from openff.pablo import STD_CCD_CACHE, ResidueDefinition, topology_from_pdb

STD_CCD_CACHE.auto_download = True

topology = topology_from_pdb(
    "7KI0_prepared.pdb",
    # Semaglutide's second NCAA is modeled in this PDB as a lysine bonded to a KUT residue
    residue_library=STD_CCD_CACHE.with_crosslink(
        residues=["LYS", "KUT"],
        linking_atoms=["NZ", "C33"],
        leaving_atoms=[["HZ2"], ["H61"]],
        bond_order=1,
    ),
    # This membrane was prepared with OpenMM, which does not use standard PDB residue names for POPC
    additional_definitions=[
        ResidueDefinition.anon_from_smiles(
            r"CCCCCCCCCCCCCCCC(=O)OC[C@H](COP(=O)([O-])OCC[N+](C)(C)C)OC(=O)CCCCCCC/C=C\CCCCCCCC"
        )
    ],
)

Semaglutide on its own:

In [ ]:
topology.molecule(0).visualize("nglview")

The full system: receptor, semaglutide, membrane, water, and ions.

In [ ]:
w = topology.visualize()
w.clear_representations()
w.add_representation("cartoon", selection=":A", color="#990000")
w.add_representation("cartoon", selection=":B", color="#FF7733")
w.add_representation("licorice", selection="not water")
w.add_representation("unitcell")
w.add_spacefill(selection="water or ion", opacity=0.05)
w

Shift everything so the protein sits in the middle of the box. The box is periodic, so this doesn't change the physics. It only makes the picture easier to read.

In [ ]:
topology.set_positions(topology.get_positions() + topology.box_vectors.sum(axis=0) / 2)

**Parametrize: the same pattern as Part 4.**

- **Lipids** (found by matching POPC's chemical formula) get **Amber Lipid17** through OpenMM
- **Protein, semaglutide, water, and ions** get the **Rosemary alpha + OPC3**
- **Combine** the two Interchanges

In [ ]:
import openmm
from openff.interchange import Interchange
from openff.toolkit import ForceField, Molecule, Topology

# Split out the POPC molecules into their own topology
popc = Molecule.from_smiles(
    r"CCCCCCCCCCCCCCCC(=O)OC[C@H](COP(=O)([O-])OCC[N+](C)(C)C)OC(=O)CCCCCCC/C=C\CCCCCCCC",
    allow_undefined_stereo=True,
)
lipid_molecules = []
prot_lig_solv_molecules = []
for mol in topology.molecules:
    if mol.hill_formula == popc.hill_formula:
        lipid_molecules.append(mol)
    else:
        prot_lig_solv_molecules.append(mol)
lipid_top = Topology.from_molecules(lipid_molecules)
prot_lig_solv_top = Topology.from_molecules(prot_lig_solv_molecules)

lipid_top.box_vectors = topology.box_vectors
prot_lig_solv_top.box_vectors = topology.box_vectors

# Prepare the protein, ligand, and solvent with the Rosemary alpha
rosemary_alpha_opc_ff = ForceField("openff_no_water-3.0.0-alpha0.offxml", "opc3.offxml")
prot_lig_solv_interchange = rosemary_alpha_opc_ff.create_interchange(prot_lig_solv_top)

# Prepare the lipid Interchange via OpenMM
lipid_ff = openmm.app.ForceField("amber14/lipid17.xml")
system = lipid_ff.createSystem(
    lipid_top.to_openmm(),
    nonbondedMethod=openmm.app.PME,
    nonbondedCutoff=9 * openmm.unit.angstrom,
    switchDistance=8 * openmm.unit.angstrom,
    constraints=openmm.app.HBonds,
)
lipid_interchange = Interchange.from_openmm(system, lipid_top)

# Combine the two Interchanges
combined_interchange = prot_lig_solv_interchange.combine(lipid_interchange)

Simulate 10,000 steps × 4 fs = **40 ps** at 1 bar. The membrane barostat lets the box resize in the membrane plane independently of the direction perpendicular to it.

> **Watch for:** semaglutide (orange, atoms shown) moving in the receptor (dark red), with the membrane shown as translucent spheres.

In [ ]:
w = simulate_and_visualize(
    combined_interchange,
    steps=10_000,
    report_interval=100,
    hydrogen_mass=3 * HYDROGEN_MASS,  # HMR for larger step size
    step_size=Quantity(4, "fs"),
    pressure=Quantity(1, "bar"),
    membrane_surface_tension=Quantity(200, "bar nm"),
)
w.clear_representations()
w.add_representation("cartoon", selection=":C", color="#990000")
w.add_representation("cartoon", selection=":A", color="#FF7733")
w.add_representation("licorice", selection=":A")
w.add_representation("spacefill", selection="POP", opacity=0.2)
w.add_representation("unitcell")
w

## Wrap-up

We just simulated a modern blockbuster, post-translationally modified peptide drug in its biological context, using an **entirely free, open-source force field and software stack**.

**Key takeaways**

1. **Same three steps every time:** topology → parametrize → simulate
2. **Chemistry first:** parameters come from chemical patterns, not atom types
3. **Next-gen PDB loader:** gets complete chemistry out of PDB files, can be taught new residues and crosslinks, and errors rather than guesses
4. **Rosemary:** one force field for proteins, peptides, small molecules, and modifications

**Resources**

- This workshop: https://github.com/openforcefield/2026-virtual-workshops/tree/main/ptm
- OpenFF documentation: https://docs.openforcefield.org